### Configure parameters

In [0]:
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "pb_threat_intelligence")

In [0]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
run_id = dbutils.widgets.get("run_id").strip()

In [0]:
audit_table = f"{catalog}.{schema}.feed_count_audit"
observations_table = f"{catalog}.{schema}.feed_observations"
output_table = f"{catalog}.{schema}.domains_classification_current"

### Check for failed records

In [0]:
# check if we have any failed status
failed_count = spark.sql(f"""
    WITH latest_records AS (
        SELECT record_count, row_number() over (ORDER BY logged_at DESC) rn
        FROM {audit_table}
        WHERE run_id = '{run_id}'
        AND status = 'failed'
    )
    SELECT sum(record_count) FROM latest_records WHERE rn = 1
""").collect()[0][0]

if failed_count:
    raise Exception(f"Found failed records in {run_id} run.")

### Reconcile counts

In [0]:
# number of records parsed from all 4 input files
input_count = spark.sql(f"""
    WITH latest_records AS (
        SELECT record_count, row_number() over (PARTITION BY feed_name ORDER BY logged_at DESC) rn
        FROM {audit_table}
        WHERE run_id = '{run_id}'
        AND stage = 'parsed'
        AND status = 'passed'
    )
    SELECT sum(record_count) FROM latest_records WHERE rn = 1
""").collect()[0][0]

print(f"With {run_id} we have {input_count:,} source records.")

# number of filltered out records
filtered_out_rows = spark.sql(f"""
    WITH latest_records AS (
        SELECT *, row_number() over (PARTITION BY feed_name, status ORDER BY logged_at DESC) rn
        FROM {audit_table}
        WHERE run_id = '{run_id}'
        AND stage = 'filtered out'
    )
    SELECT feed_name, status, sum(record_count) as record_count
    FROM latest_records
    WHERE rn = 1
    GROUP BY feed_name, status
""").collect()

filtered_out_count = 0

for row in filtered_out_rows:
    feed_name, status, record_count = row
    print(f"Filtered out {record_count:,} records, because was classified as {status} (in/between {feed_name})")
    filtered_out_count += record_count

print(f"Total filtered out records: {filtered_out_count:,}")

# number of records in the output table
output_count = spark.sql(f"""
    SELECT count(*) FROM {output_table}
""").collect()[0][0]

print(f"Output table has {output_count:,} records.")

In [0]:
reconciled_counts = input_count - filtered_out_count - output_count

### Validate reconciliation

In [0]:
# check if we have any reconciled counts
# in case when input_counts are not matching with output_counts + filtered_out_counts
# we rollback the current_state table to its previous version

if reconciled_counts != 0:
    previous_version = spark.sql(f"""DESCRIBE HISTORY {output_table}""").selectExpr("max(version) as version").collect()[0][0] - 1
    spark.sql(f"RESTORE TABLE {output_table} TO VERSION AS OF {previous_version}")
    raise Exception(f"Reconciled counts are not equal to zero: {reconciled_counts}. Rolled-back {output_table} to its previous version")
else:
    print("Reconciled counts are equal to zero.")